[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/monacofj/moeabench/blob/main/examples/example_19.ipynb)

# Example 19: Hypervolume and Ordinal Hypervolume

This notebook compares two experiments using two complementary convergence measures:

- **Hypervolume (HV)** preserves metric distances in objective space.
- **Ordinal Hypervolume (OHV)** replaces consecutive distinct reference levels with unit ordinal steps.

Both experiments use the same explicit reference context. This is essential: HV needs common normalization bounds, while OHV needs one common ordinal lattice.

In [ ]:
# Install moeabench from GitHub
!pip install --quiet git+https://github.com/monacofj/moeabench.git

In [ ]:
from moeabench import mb

mb.system.version()

## 1. Configure and run the experiments

We compare NSGA-II and NSGA-III on the same three-objective DTLZ2 problem. Three repetitions provide a distribution at each generation, so the plots can display mean trajectories and variability bands. Fixed seeds make the example reproducible.

In [ ]:
exp1 = mb.experiment()
exp1.name = "NSGA-II"
exp1.mop = mb.mops.DTLZ2(M=3)
exp1.moea = mb.moeas.NSGA2(population=60, generations=40, seed=7)

exp2 = mb.experiment()
exp2.name = "NSGA-III"
exp2.mop = mb.mops.DTLZ2(M=3)
exp2.moea = mb.moeas.NSGA3(population=60, generations=40, seed=19)

exp1.run(repeat=3)
exp2.run(repeat=3)

## 2. Compute HV and OHV with one shared reference

`reference = [exp1, exp2]` pools the final fronts of both experiments. For conventional HV, those fronts define one ideal/nadir normalization context. For OHV, they define the sorted distinct levels of one fixed ordinal coordinate system.

The ordinal lattice is built once and reused for every generation and run. It is never recomputed generation by generation.

In [ ]:
reference = [exp1, exp2]

hv1 = mb.metrics.hv(exp1, ref=reference, mode="exact")
hv2 = mb.metrics.hv(exp2, ref=reference, mode="exact")
ohv1 = mb.metrics.ohv(exp1, ref=reference, mode="exact")
ohv2 = mb.metrics.ohv(exp2, ref=reference, mode="exact")

## 3. Compare the convergence plots

The left plot is distance-sensitive. The right plot retains ordering and Pareto-dominance information while discarding the magnitude of differences between consecutive reference values. Dashed envelopes show the minimum and maximum run values; shaded regions show one standard deviation around the mean.

In [ ]:
mb.view.history(
    hv1, hv2,
    title="Conventional Hypervolume (HV)",
    show_bounds=True,
)
mb.view.history(
    ohv1, ohv2,
    title="Ordinal Hypervolume (OHV)",
    show_bounds=True,
)

## 4. Inspect the metric reports

The HV report describes its normalization geometry. The OHV report instead describes the ordinal level counts, ordinal reference point, ordinal box volume, and `OHV/OBox`. The raw OHV values remain ordinal dominated volumes; `OHV/OBox` is diagnostic only.

In [ ]:
hv1.report()
ohv1.report()

## Interpretation

A difference visible in HV but attenuated in OHV is primarily associated with metric distances between objective values. A difference retained by OHV reflects ordering and coverage of levels in the common ordinal reference lattice. Neither metric replaces the other: together they separate geometric progress from ordinal progress.

The same shared context can also be established automatically for a direct plot of raw experiments with `mb.view.history(exp1, exp2, metric=mb.metrics.ohv)`. Here we calculate the metric matrices explicitly so their diagnostics and reports remain available for inspection.